# KOWAS-EPI 추가 EDA 시각화 (2단계 확장)

`01_fitting_period_eda.ipynb`(fold별 검정 A/B·교차상관·기술통계·plant_meta 탐색)에 이어,
README에 명시된 다른 데이터 특성들(지역별 스케일 차이, 결측 구조, wow_change_rate 이상치,
강수-농도 원산점도, pop_sampled 중복집계, 급증 경보 클래스 불균형)을 추가로 시각화한다.

**가정 및 전제**
- 기술통계·탐색 목적이므로 fitting 가능한 전체 구간(가장 늦은 fold, `train_until=2026-W25`)까지의
  데이터를 사용한다 — `01_fitting_period_eda.ipynb`의 §1 기술통계·§4 plant_meta 탐색과 동일 원칙.
- 라벨(`target_alert_t2`)을 다루는 그림은 `target_week_t2 ≤ 2026-W25`인 행만 사용한다
  (`evaluation.py`의 `known_at()`과 동일 누수 차단 원칙).
- 평가 창(2026-W27 이후) 데이터는 어떤 그림에서도 사용하지 않는다.

In [1]:
import datetime
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.font_manager as fm
_font_path = str(Path.home() / ".local/share/fonts/NanumGothic-Regular.ttf")
fm.fontManager.addfont(_font_path)
matplotlib.rcParams["font.family"] = [fm.FontProperties(fname=_font_path).get_name(), "DejaVu Sans"]
matplotlib.rcParams["axes.unicode_minus"] = False
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

ROOT = Path("/home/geon1/github/kowas-epi-forecast")
KOWAS = ROOT / "KOWAS-EPI"
FIG = ROOT / "figures"
pd.set_option("display.max_columns", None)


def parse(k):
    y, w = k.split("-W")
    return (int(y), int(w))


def le(a, b):
    return parse(a) <= parse(b)


def to_date(k):
    y, w = parse(k)
    return datetime.date.fromisocalendar(y, w, 1)


T_MAX = "2026-W25"  # 가장 늦은 fold(2026-Q3)의 train_until — fitting 가능 최대 구간

panel = pd.read_csv(KOWAS / "kowas_epi_panel.csv", dtype=str, keep_default_na=False)
national = pd.read_csv(KOWAS / "national_weekly.csv", dtype=str, keep_default_na=False)

p = panel[panel["date_week"].apply(lambda w: le(w, T_MAX))].copy()
for c in ["conc_mean", "conc_3wk_avg", "wow_change_rate", "precip_mm", "temp_avg", "pop_served", "pop_sampled"]:
    p[c] = pd.to_numeric(p[c].replace("", np.nan))
p["week_no"] = p["date_week"].str.split("-W").str[1].astype(int)
print(f"fitting rows (<= {T_MAX}): {len(p)}")

fitting rows (<= 2026-W25): 2856


## 05. 지역별 `conc_mean` 분포 (boxplot, 로그축)

README §5-2가 "시·도 간 100배 이상 차이"라고 서술한 스케일 차이를, 지역별 중앙값 순으로
정렬한 boxplot으로 직관적으로 보여준다.

In [2]:
order = p.groupby("region")["conc_mean"].median().sort_values().index.tolist()
data = [p.loc[p["region"] == r, "conc_mean"].dropna().values for r in order]

fig, ax = plt.subplots(figsize=(11, 5))
ax.boxplot(data, tick_labels=order, showfliers=True, flierprops=dict(marker=".", markersize=3, alpha=0.4))
ax.set_yscale("log")
ax.set_ylabel("conc_mean (copies/mL, log scale)")
ax.set_xlabel("시·도 (중앙값 오름차순)")
ax.set_title(f"지역별 conc_mean 분포 (fitting 구간 ~{T_MAX})")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(FIG / "05_regional_conc_boxplot.png", dpi=130)
plt.show()
plt.close()
med_min, med_max = p.groupby("region")["conc_mean"].median().min(), p.groupby("region")["conc_mean"].median().max()
print("median range ratio (max/min):", med_max / med_min)

median range ratio (max/min): 28.09047920731862


## 06. `conc_mean` 결측 패턴 히트맵 (지역 × 주차)

README §5-1의 결측 구조(연말 미보고 `no_source_row`, 부분 미측정 `no_measurement`)를
지역 × 주차 그리드로 시각화한다.

In [3]:
reason_code = {"": 0, "no_measurement": 1, "no_source_row": 2}
p["reason_code"] = p["missing_reason"].map(reason_code)
weeks_sorted = sorted(p["date_week"].unique().tolist(), key=parse)
regions_sorted = sorted(p["region"].unique().tolist())
mat = p.pivot(index="region", columns="date_week", values="reason_code").reindex(index=regions_sorted, columns=weeks_sorted)

cmap = ListedColormap(["#dfe7f2", "#f4a261", "#e76f51"])
fig, ax = plt.subplots(figsize=(16, 5))
im = ax.imshow(mat.values, aspect="auto", cmap=cmap, vmin=0, vmax=2)
ax.set_yticks(range(len(regions_sorted)))
ax.set_yticklabels(regions_sorted, fontsize=8)
xticks_idx = list(range(0, len(weeks_sorted), 13))
ax.set_xticks(xticks_idx)
ax.set_xticklabels([weeks_sorted[i] for i in xticks_idx], rotation=45, ha="right", fontsize=8)
ax.set_title(f"conc_mean 결측 패턴 (지역 x 주차, ~{T_MAX})")
handles = [Patch(color="#dfe7f2", label="관측됨"), Patch(color="#f4a261", label="no_measurement"),
           Patch(color="#e76f51", label="no_source_row")]
ax.legend(handles=handles, loc="upper center", bbox_to_anchor=(0.5, -0.25), ncol=3, frameon=False)
plt.tight_layout()
plt.savefig(FIG / "06_missingness_heatmap.png", dpi=130)
plt.show()
plt.close()
print("missing rows:", (p["reason_code"] > 0).sum())

missing rows: 189


## 07. `wow_change_rate` 이상치 분포

README §5-3이 명시한 극단적 이상치(−4,731~9,639)를 symlog 히스토그램(전체)과
중심부 확대 히스토그램으로 함께 보여준다.

In [4]:
wow = p["wow_change_rate"].dropna()
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].hist(wow, bins=200)
axes[0].set_yscale("symlog")
axes[0].set_xlabel("wow_change_rate")
axes[0].set_ylabel("빈도 (symlog)")
axes[0].set_title(f"전체 분포 (n={len(wow)}, 범위 {wow.min():.0f}~{wow.max():.0f})")

clip = wow[(wow > -20) & (wow < 20)]
axes[1].hist(clip, bins=100, color="tab:blue")
axes[1].set_xlabel("wow_change_rate (−20~20 구간 확대)")
axes[1].set_title(f"중심부 확대 (n={len(clip)}, {len(clip)/len(wow)*100:.1f}%가 이 구간)")
fig.suptitle("wow_change_rate 이상치 (README §5-3)")
plt.tight_layout()
plt.savefig(FIG / "07_wow_change_rate_outliers.png", dpi=130)
plt.show()
plt.close()
print("extreme(|x|>100):", ((wow > 100) | (wow < -100)).sum())

extreme(|x|>100): 3


## 08. 강수-농도희석 원산점도

검정 A(부분상관)는 지역·계절 통제 후 잔차 상관이라 추상적이다. 여기서는 통제 없이
`log(1+precip_mm)` vs `log10(conc_mean/conc_3wk_avg)` 원본 산점도로 "왜 상관이 거의 0인지"를
시각적으로 보여준다(마지막 fold 기준, 데이터가 가장 많음).

In [5]:
sub = p[(p["conc_mean"] > 0) & (p["conc_3wk_avg"] > 0) & p["precip_mm"].notna()].copy()
sub["y"] = np.log10(sub["conc_mean"] / sub["conc_3wk_avg"])
sub["x"] = np.log(1 + sub["precip_mm"])
coef = np.polyfit(sub["x"], sub["y"], 1)
xs = np.linspace(sub["x"].min(), sub["x"].max(), 50)

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(sub["x"], sub["y"], s=8, alpha=0.25, color="tab:blue")
ax.plot(xs, coef[0] * xs + coef[1], color="tab:red", lw=2, label=f"전체 회귀선 (기울기={coef[0]:.4f})")
ax.set_xlabel("log(1 + precip_mm)")
ax.set_ylabel("log10(conc_mean / conc_3wk_avg)")
r_raw = np.corrcoef(sub["x"], sub["y"])[0, 1]
ax.set_title(f"강수 vs 농도희석 원산점도 (n={len(sub)}, 단순상관 r={r_raw:.3f})")
ax.legend()
plt.tight_layout()
plt.savefig(FIG / "08_precip_dilution_scatter.png", dpi=130)
plt.show()
plt.close()
print("n=", len(sub), "raw r=", r_raw)

n= 2667 raw r= 0.011216443670622944


## 09. `pop_sampled` vs `pop_served` — 중복집계 이슈

README §5-5가 지적한, 채취 처리장 처리인구 합(`pop_sampled`)이 시·도 고정 처리인구(`pop_served`)를
초과하는 201행(서울 110·대전 91)을 y=x 기준선과 함께 강조한다.

In [6]:
sub9 = p[p["pop_sampled"].notna()].copy()
over = sub9["pop_sampled"] > sub9["pop_served"]
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(sub9.loc[~over, "pop_served"], sub9.loc[~over, "pop_sampled"], s=10, alpha=0.4, color="tab:blue",
           label=f"pop_sampled ≤ pop_served (n={(~over).sum()})")
ax.scatter(sub9.loc[over, "pop_served"], sub9.loc[over, "pop_sampled"], s=14, alpha=0.7, color="tab:red",
           label=f"pop_sampled > pop_served (n={over.sum()})")
lims = [0, max(sub9["pop_served"].max(), sub9["pop_sampled"].max()) * 1.05]
ax.plot(lims, lims, "k--", lw=1, label="y = x")
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_xlabel("pop_served")
ax.set_ylabel("pop_sampled")
ax.set_title("pop_sampled vs pop_served — 중복집계 이슈 (README §5-5)")
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(FIG / "09_pop_sampled_vs_served.png", dpi=130)
plt.show()
plt.close()
print("over count:", over.sum(), "regions:", sub9.loc[over, "region"].value_counts().to_dict())

over count: 201 regions: {'서울': 110, '대전': 91}


## 10. `target_alert_t2` 양성 비율 — 지역별 · 분기별

분류 과제의 클래스 불균형(README 전체 양성률 30.6%)이 지역별로는 완만하게(약 0.20~0.36),
분기(목표주 기준)별로는 훨씬 뚜렷하게(약 0.07~0.48) 갈리는지 확인한다.
`target_week_t2 ≤ 2026-W25`인 라벨만 사용(누수 차단).

In [7]:
lbl = panel[panel["target_week_t2"].apply(lambda w: le(w, T_MAX) if w else False)].copy()
lbl = lbl[lbl["target_alert_t2"] != ""]
lbl["is_pos"] = (lbl["target_alert_t2"] == "1").astype(int)
lbl["tdate"] = lbl["target_week_t2"].apply(to_date)
lbl["quarter"] = lbl["tdate"].apply(lambda d: f"{d.year}-Q{(d.month-1)//3+1}")

reg_rate = lbl.groupby("region")["is_pos"].mean().sort_values()
q_order = sorted(lbl["quarter"].unique(), key=lambda s: (int(s[:4]), int(s[-1])))
q_rate = lbl.groupby("quarter")["is_pos"].agg(rate="mean", n="count").reindex(q_order)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
axes[0].bar(reg_rate.index, reg_rate.values, color="tab:blue")
axes[0].axhline(lbl["is_pos"].mean(), color="k", ls="--", lw=1, label=f"전체 평균 {lbl['is_pos'].mean():.3f}")
axes[0].set_ylabel("target_alert_t2 양성 비율")
axes[0].set_title("지역별 양성 비율")
axes[0].legend(fontsize=8)
plt.setp(axes[0].get_xticklabels(), rotation=45, ha="right")

axes[1].bar(q_rate.index, q_rate["rate"], color="tab:orange")
axes[1].axhline(lbl["is_pos"].mean(), color="k", ls="--", lw=1)
axes[1].set_ylabel("target_alert_t2 양성 비율")
axes[1].set_title("분기(목표주 기준)별 양성 비율 — 뚜렷한 계절성")
plt.setp(axes[1].get_xticklabels(), rotation=45, ha="right", fontsize=8)
fig.suptitle(f"급증 경보(target_alert_t2) 양성 비율, n={len(lbl)} (target_week_t2 ≤ {T_MAX})")
plt.tight_layout()
plt.savefig(FIG / "10_alert_positive_rate.png", dpi=130)
plt.show()
plt.close()
print("region range:", reg_rate.min(), reg_rate.max(), " quarter range:", q_rate["rate"].min(), q_rate["rate"].max())

region range: 0.1962025316455696 0.3584905660377358  quarter range: 0.0748663101604278 0.4751131221719457


## 요약

6개 그림 모두 README에 명시된 데이터 특성(스케일 차이, 결측 구조, 증감률 이상치, 강수 희석 부재,
처리인구 중복집계) 또는 새로 발견한 패턴(급증 경보의 뚜렷한 분기별 계절성)을 시각적으로 보여준다.
상세 해석은 `figures/README.md`를 참고.